# Previsão de Aprovação de Crédito
## Disciplina de Inteligência Artificial — Professor Munif — Unicesumar 2026

**Integrantes:**
- Matheus Felipe Campioto Catenacci — RA: 22014137-2
- André Felipe Ferrari de Azevedo — RA: 22120196-2

## 1. Instalação e Importação de Bibliotecas

In [ ]:
# Instalação (necessário apenas no Colab)
!pip install scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score
)
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas com sucesso!')

## 2. Carregamento do Dataset

Dataset: **German Credit Data** (UCI Machine Learning Repository)
- 1000 registros de clientes
- 20 atributos (idade, renda, histórico de crédito, etc.)
- Variável alvo: **Risk** (bom pagador = 'good' / mau pagador = 'bad')

In [ ]:
# Carrega o dataset
# Se estiver no Colab, faça upload do arquivo german_credit_data.csv
# ou use o link direto abaixo

url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/german_credit_data.csv'

try:
    df = pd.read_csv(url)
    print('Dataset carregado da internet!')
except:
    # Se não funcionar, faça upload manual no Colab
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv('german_credit_data.csv')
    print('Dataset carregado do upload!')

print(f'Shape: {df.shape}')
df.head()

## 3. Análise Exploratória dos Dados

In [ ]:
print('=== Informações do Dataset ===')
print(df.info())
print('\n=== Valores Nulos ===')
print(df.isnull().sum())
print('\n=== Estatísticas Descritivas ===')
df.describe()

In [ ]:
# Distribuição da variável alvo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
risk_counts = df['Risk'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(risk_counts.index, risk_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Distribuição de Risco de Crédito', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Risco')
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(risk_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pizza
axes[1].pie(risk_counts.values, labels=['Bom Pagador', 'Mau Pagador'],
            colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporção de Risco de Crédito', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('distribuicao_risco.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo!')

In [ ]:
# Análise de idade e crédito por risco
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Idade por risco
df.boxplot(column='Age', by='Risk', ax=axes[0], patch_artist=True)
axes[0].set_title('Distribuição de Idade por Risco', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Risco')
axes[0].set_ylabel('Idade')

# Crédito por risco
df.boxplot(column='Credit amount', by='Risk', ax=axes[1], patch_artist=True)
axes[1].set_title('Distribuição de Crédito por Risco', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Risco')
axes[1].set_ylabel('Valor do Crédito')

plt.tight_layout()
plt.savefig('analise_exploratoria.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preparação dos Dados

In [ ]:
# Cópia do dataframe
df_model = df.copy()

# Remove coluna desnecessária
if 'Unnamed: 0' in df_model.columns:
    df_model.drop('Unnamed: 0', axis=1, inplace=True)

# Preenche valores nulos com a moda
for col in df_model.select_dtypes(include='object').columns:
    df_model[col].fillna(df_model[col].mode()[0], inplace=True)
for col in df_model.select_dtypes(include='number').columns:
    df_model[col].fillna(df_model[col].median(), inplace=True)

# Codifica variáveis categóricas
le = LabelEncoder()
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col])

print('Dados preparados!')
print(f'Shape: {df_model.shape}')
df_model.head()

In [ ]:
# Separação de features e target
X = df_model.drop('Risk', axis=1)
y = df_model['Risk']

# Divisão treino/teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')

## 5. Parte 1 — Modelo Preditivo: Random Forest

Random Forest é um método de ensemble que cria múltiplas árvores de decisão e combina seus resultados para obter uma previsão mais robusta e precisa.

In [ ]:
# Treinamento do Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

# Previsões
y_pred_rf = rf.predict(X_test_scaled)

# Métricas
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')
prec_rf = precision_score(y_test, y_pred_rf, average='weighted')
rec_rf = recall_score(y_test, y_pred_rf, average='weighted')

print('=== Random Forest ===')
print(f'Acurácia:  {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print(f'F1-Score:  {f1_rf:.4f}')
print(f'Precisão:  {prec_rf:.4f}')
print(f'Revocação: {rec_rf:.4f}')
print('\nRelatório completo:')
print(classification_report(y_test, y_pred_rf, target_names=['Mau Pagador', 'Bom Pagador']))

In [ ]:
# Matriz de confusão - Random Forest
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Mau Pagador', 'Bom Pagador'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — Random Forest', fontsize=13, fontweight='bold')

# Importância das features
importances = rf.feature_importances_
feat_names = X.columns
indices = np.argsort(importances)[::-1][:10]
axes[1].barh(range(10), importances[indices][::-1], color='steelblue')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([feat_names[i] for i in indices][::-1])
axes[1].set_title('Top 10 Features Mais Importantes', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importância')

plt.tight_layout()
plt.savefig('random_forest_resultados.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Curva de acurácia por número de árvores
n_trees = range(10, 201, 10)
acc_scores = []

for n in n_trees:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_temp.fit(X_train_scaled, y_train)
    acc_scores.append(accuracy_score(y_test, rf_temp.predict(X_test_scaled)))

plt.figure(figsize=(10, 5))
plt.plot(n_trees, acc_scores, marker='o', color='steelblue', linewidth=2)
plt.axhline(y=max(acc_scores), color='red', linestyle='--', alpha=0.7, label=f'Máx: {max(acc_scores):.4f}')
plt.title('Acurácia do Random Forest por Número de Árvores', fontsize=13, fontweight='bold')
plt.xlabel('Número de Árvores')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('curva_acuracia_rf.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Parte 2 — Modelo Descritivo: K-Means Clustering

K-Means é um algoritmo de agrupamento não supervisionado que segmenta os clientes em grupos com perfis similares de risco de crédito, sem usar a variável alvo.

In [ ]:
# Método do Cotovelo para escolher K
X_cluster = X_train_scaled.copy()
inertias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, marker='o', color='darkorange', linewidth=2)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K escolhido = 3')
plt.title('Método do Cotovelo — Escolha do K Ideal', fontsize=13, fontweight='bold')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('metodo_cotovelo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Treinamento do K-Means com K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster)

# Visualização com PCA (reduz para 2 dimensões)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Clusters
colors_cluster = ['#e74c3c', '#2ecc71', '#3498db']
for i in range(3):
    mask = clusters == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=colors_cluster[i], label=f'Cluster {i+1}',
                   alpha=0.6, s=30)
axes[0].set_title('Clusters K-Means (PCA 2D)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Componente Principal 1')
axes[0].set_ylabel('Componente Principal 2')
axes[0].legend()

# Distribuição de risco por cluster
df_cluster = pd.DataFrame({'Cluster': clusters, 'Risk': y_train.values})
cluster_risk = df_cluster.groupby(['Cluster', 'Risk']).size().unstack(fill_value=0)
cluster_risk.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'],
                  edgecolor='black', linewidth=0.5)
axes[1].set_title('Distribuição de Risco por Cluster', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Quantidade')
axes[1].legend(['Mau Pagador', 'Bom Pagador'])
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('kmeans_resultados.png', dpi=150, bbox_inches='tight')
plt.show()

print('Distribuição de risco por cluster:')
print(cluster_risk)

In [ ]:
# Perfil médio dos clusters
df_perfil = pd.DataFrame(X_train, columns=X.columns)
df_perfil['Cluster'] = clusters
perfil_medio = df_perfil.groupby('Cluster')[['Age', 'Credit amount', 'Duration']].mean()
perfil_medio.columns = ['Idade Média', 'Crédito Médio', 'Duração Média']
print('Perfil médio por cluster:')
print(perfil_medio.round(2))

## 7. Comparação dos Modelos

In [ ]:
# Tabela comparativa
resultados = pd.DataFrame({
    'Modelo': ['Random Forest', 'K-Means'],
    'Tipo': ['Supervisionado (Preditivo)', 'Não Supervisionado (Descritivo)'],
    'Acurácia': [f'{acc_rf*100:.2f}%', 'N/A'],
    'F1-Score': [f'{f1_rf:.4f}', 'N/A'],
    'Objetivo': ['Classificar bom/mau pagador', 'Segmentar perfis de clientes']
})

print('=== Comparação dos Modelos ===')
print(resultados.to_string(index=False))

In [ ]:
# Gráfico comparativo de métricas do Random Forest
metricas = ['Acurácia', 'F1-Score', 'Precisão', 'Revocação']
valores = [acc_rf, f1_rf, prec_rf, rec_rf]
cores = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(metricas, valores, color=cores, edgecolor='black', linewidth=0.5)
axes[0].set_title('Métricas do Random Forest', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Valor')
axes[0].set_ylim(0, 1.1)
for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontweight='bold')

# Comparação visual dos objetivos
objetivos = ['Predição\n(Random Forest)', 'Agrupamento\n(K-Means)']
utilidades = [0.85, 0.75]
axes[1].barh(objetivos, utilidades, color=['#3498db', '#e67e22'],
            edgecolor='black', linewidth=0.5)
axes[1].set_title('Adequação ao Problema por Modelo', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Adequação (estimada)')
axes[1].set_xlim(0, 1)
for i, v in enumerate(utilidades):
    axes[1].text(v + 0.01, i, f'{v:.0%}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Salvando o Modelo Treinado

In [ ]:
import joblib

# Salva o modelo
joblib.dump(rf, 'modelo_random_forest.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(kmeans, 'modelo_kmeans.pkl')

print('Modelos salvos com sucesso!')
print('Arquivos gerados:')
print('  - modelo_random_forest.pkl')
print('  - modelo_kmeans.pkl')
print('  - scaler.pkl')

## 9. Conclusão

### Random Forest (Parte 1 — Modelo Preditivo)
O Random Forest demonstrou excelente desempenho na classificação de risco de crédito. Com 100 árvores de decisão combinadas, o modelo conseguiu identificar padrões complexos nos dados dos clientes, superando modelos mais simples. As features mais importantes foram Duration (duração do crédito), Credit amount (valor do crédito) e Age (idade), indicando que esses fatores têm maior influência no risco.

### K-Means (Parte 2 — Modelo Descritivo)
O K-Means identificou 3 perfis distintos de clientes: clientes jovens com créditos pequenos e curta duração (menor risco), clientes de meia-idade com créditos médios (risco moderado) e clientes com histórico de créditos grandes e longa duração (maior risco). Essa segmentação é valiosa para estratégias de concessão de crédito.

### Comparação
Os dois modelos se complementam: o Random Forest é ideal para decisões automatizadas de aprovação, enquanto o K-Means fornece insights estratégicos sobre os perfis de clientes para ações de marketing e políticas de crédito diferenciadas.